In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import sys
from urllib.parse import urljoin

# Set headers to mimic a browser request
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Initialize data storage
data = {
    'Title': [],
    'URL': [],
    'Date': []
}

# Set maximum data size (500 MB in bytes)
MAX_SIZE_BYTES = 500 * 1024 * 1024  # 500 MB

def get_data_size(df):
    """Estimate the size of the DataFrame in bytes."""
    csv_buffer = df.to_csv(index=False)
    return sys.getsizeof(csv_buffer)

def scrape_article_data(url):
    """Scrape title, URL, and date from an article page."""
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Extract title
        title_tag = soup.find('h1', class_='entry-title')
        title = title_tag.get_text(strip=True) if title_tag else 'No Title'
        
        # Extract date
        date_tag = soup.find('time', class_='entry-date')
        date = date_tag.get_text(strip=True) if date_tag else 'No Date'
        
        return title, url, date
    except requests.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return None, None, None

def main():
    base_url = 'https://datascienceplus.com'
    try:
        # Fetch the main page
        response = requests.get(base_url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Find article links
        articles = soup.find_all('article')
        print(f"Found {len(articles)} articles on the main page.")
        
        for article in articles:
            # Check data size before proceeding
            df = pd.DataFrame(data)
            if get_data_size(df) >= MAX_SIZE_BYTES:
                print("Data size limit of 500 MB reached. Stopping scraping.")
                break
            
            # Extract article URL
            link_tag = article.find('a', class_='entry-title-link')
            if not link_tag or 'href' not in link_tag.attrs:
                continue
            article_url = urljoin(base_url, link_tag['href'])
            
            # Scrape article data
            title, url, date = scrape_article_data(article_url)
            if title and url and date:
                data['Title'].append(title)
                data['URL'].append(url)
                data['Date'].append(date)
                print(f"Scraped: {title}")
        
        # Create DataFrame
        df = pd.DataFrame(data)
        print(f"Total articles scraped: {len(df)}")
        
        # Save to CSV
        output_file = 'datascienceplus_articles.csv'
        df.to_csv(output_file, index=False)
        print(f"Data saved to {output_file}")
        
        # Verify file size
        file_size = os.path.getsize(output_file) / (1024 * 1024)  # Size in MB
        print(f"CSV file size: {file_size:.2f} MB")
        
    except requests.RequestException as e:
        print(f"Error fetching main page: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

if __name__ == '__main__':
    main()

Found 9 articles on the main page.
Total articles scraped: 0
Data saved to datascienceplus_articles.csv
CSV file size: 0.00 MB


In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import sys
from urllib.parse import urljoin
import time
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Set headers to mimic a browser request
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Initialize data storage
data = {
    'Title': [],
    'URL': [],
    'Date': []
}

# Set maximum data size (500 MB in bytes)
MAX_SIZE_BYTES = 500 * 1024 * 1024  # 500 MB

def get_data_size(df):
    """Estimate the size of the DataFrame in bytes."""
    csv_buffer = df.to_csv(index=False)
    return sys.getsizeof(csv_buffer)

def scrape_article_data(url):
    """Scrape title, URL, and date from an article page."""
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Flexible title extraction (try multiple tags/classes)
        title_tag = soup.find('h1', class_='entry-title') or soup.find('h1')
        title = title_tag.get_text(strip=True) if title_tag else 'No Title'
        
        # Flexible date extraction
        date_tag = soup.find('time', class_='entry-date') or soup.find('time') or soup.find('meta', property='article:published_time')
        if date_tag:
            date = date_tag.get_text(strip=True) or date_tag.get('content', 'No Date')
        else:
            date = 'No Date'
        
        logger.info(f"Scraped article: {title} from {url}")
        return title, url, date
    except requests.RequestException as e:
        logger.error(f"Error fetching {url}: {e}")
        return None, None, None

def get_article_links(page_url, base_url):
    """Fetch article links from a given page."""
    try:
        response = requests.get(page_url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Find article links (flexible selector)
        articles = soup.find_all('article') or soup.find_all('div', class_=['post', 'article', 'entry'])
        links = []
        for article in articles:
            link_tag = article.find('a', class_='entry-title-link') or article.find('a', href=True)
            if link_tag and 'href' in link_tag.attrs:
                full_url = urljoin(base_url, link_tag['href'])
                links.append(full_url)
        return links, soup
    except requests.RequestException as e:
        logger.error(f"Error fetching {page_url}: {e}")
        return [], None

def main():
    base_url = 'https://datascienceplus.com'
    page_url = base_url
    all_links = set()  # Use set to avoid duplicates
    page_num = 1
    max_pages = 5  # Limit to avoid excessive scraping; adjust as needed
    
    while page_url and page_num <= max_pages:
        logger.info(f"Scraping page {page_num}: {page_url}")
        links, soup = get_article_links(page_url, base_url)
        logger.info(f"Found {len(links)} article links on page {page_num}")
        
        # Add unique links
        all_links.update(links)
        
        # Check for pagination (next page link)
        next_page = soup.find('a', class_='next') or soup.find('a', text='Next') or soup.find('a', rel='next')
        if next_page and 'href' in next_page.attrs:
            page_url = urljoin(base_url, next_page['href'])
            page_num += 1
            time.sleep(1)  # Delay to avoid overwhelming server
        else:
            page_url = None
        
        # Scrape article data
        for article_url in links:
            # Check data size before proceeding
            df = pd.DataFrame(data)
            if get_data_size(df) >= MAX_SIZE_BYTES:
                logger.warning("Data size limit of 500 MB reached. Stopping scraping.")
                break
            
            title, url, date = scrape_article_data(article_url)
            if title and url and date:
                data['Title'].append(title)
                data['URL'].append(url)
                data['Date'].append(date)
            time.sleep(0.5)  # Delay between article requests
        
        if get_data_size(df) >= MAX_SIZE_BYTES:
            break
    
    # Create DataFrame
    df = pd.DataFrame(data)
    logger.info(f"Total articles scraped: {len(df)}")
    
    # Save to CSV
    output_file = 'datascienceplus_articles.csv'
    if not df.empty:
        df.to_csv(output_file, index=False)
        file_size = os.path.getsize(output_file) / (1024 * 1024)  # Size in MB
        logger.info(f"Data saved to {output_file}. CSV file size: {file_size:.2f} MB")
    else:
        logger.warning("No data was scraped. CSV file not created.")
    
    # Log all found links for debugging
    logger.info(f"Total unique article links found: {len(all_links)}")
    logger.info(f"Links: {list(all_links)}")

if __name__ == '__main__':
    main()

2025-07-23 10:28:09,258 - INFO - Scraping page 1: https://datascienceplus.com
2025-07-23 10:28:10,892 - INFO - Found 9 article links on page 1
2025-07-23 10:28:12,567 - INFO - Scraped article: Linking R and Python to retrieve financial data and plot a candlestick from https://datascienceplus.com/linking-r-and-python-to-retrieve-financial-data-and-plot-a-candlestick/
2025-07-23 10:28:14,041 - INFO - Scraped article: An R alternative to pairs for -omics QC from https://datascienceplus.com/an-r-alternative-to-pairs-for-omics-qc/
2025-07-23 10:28:16,132 - INFO - Scraped article: Visualizing economic data with pretty worldmaps from https://datascienceplus.com/visualizing-economic-data-with-pretty-worldmaps/
2025-07-23 10:28:18,262 - INFO - Scraped article: How I selected my starting word for Wordle using simulations and R from https://datascienceplus.com/how-i-selected-my-starting-word-for-wordle-using-simulations-and-r/
2025-07-23 10:28:20,370 - INFO - Scraped article: Forecast using Arima

In [4]:
pip install selenium webdriver_manager


ERROR: After October 2020 you may experience errors when installing or updating packages. This is because pip will change the way that it resolves dependency conflicts.

We recommend you use --use-feature=2020-resolver to test your packages with the new resolver before it becomes the default.

fbprophet 0.7.1 requires cmdstanpy==0.9.5, which is not installed.
huggingface-hub 0.33.4 requires packaging>=20.9, but you'll have packaging 20.4 which is incompatible.
google-api-core 1.22.2 requires protobuf>=3.12.0, but you'll have protobuf 3.11.2 which is incompatible.
botocore 1.18.16 requires urllib3<1.26,>=1.20, but you'll have urllib3 2.2.3 which is incompatible.



  Attempting uninstall: urllib3
    Found existing installation: urllib3 1.25.10
    Uninstalling urllib3-1.25.10:
      Successfully uninstalled urllib3-1.25.10
  Attempting uninstall: certifi
    Found existing installation: certifi 2020.6.20
    Uninstalling certifi-2020.6.20:
      Successfully uninstalled certifi-2020.6.20
  Attempting uninstall: attrs
    Found existing installation: attrs 20.2.0
    Uninstalling attrs-20.2.0:
      Successfully uninstalled attrs-20.2.0


In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import sys
import logging
from urllib.parse import urljoin
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Set headers for requests
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Initialize data storage (updated to include full text)
data = {
    'Title': [],
    'URL': [],
    'Date': [],
    'Content': []
}

# Set maximum data size (500 MB in bytes)
MAX_SIZE_BYTES = 500 * 1024 * 1024  # 500 MB
MAX_WORDS = 1000  # Maximum words per article

def get_data_size(df):
    """Estimate the size of the DataFrame in bytes."""
    csv_buffer = df.to_csv(index=False)
    return sys.getsizeof(csv_buffer)

def scrape_api(base_url):
    """Scrape blog posts using WordPress REST API."""
    api_url = f"{base_url}/wp-json/wp/v2/posts"
    params = {'per_page': 100, 'page': 1}  # Fetch up to 100 posts per page
    scraped_count = 0
    
    try:
        while True:
            response = requests.get(api_url, headers=headers, params=params, timeout=10)
            response.raise_for_status()
            posts = response.json()
            
            if not posts:  # No more posts
                break
            
            for post in posts:
                title = post.get('title', {}).get('rendered', 'Untitled')
                url = post.get('link', '')
                date = post.get('date', 'No Date')
                # Extract content (rendered HTML, strip tags)
                content = post.get('content', {}).get('rendered', '')
                soup = BeautifulSoup(content, 'html.parser')
                paragraphs = soup.find_all('p')
                full_text = ' '.join(p.get_text(strip=True) for p in paragraphs)
                words = full_text.split()
                content = ' '.join(words[:MAX_WORDS])
                
                data['Title'].append(title)
                data['URL'].append(url)
                data['Date'].append(date)
                data['Content'].append(content)
                scraped_count += 1
                logger.info(f"Scraped via API: {title}")
                
                # Check data size
                df = pd.DataFrame(data)
                if get_data_size(df) >= MAX_SIZE_BYTES:
                    logger.warning("Data size limit of 500 MB reached. Stopping API scraping.")
                    return scraped_count
            
            params['page'] += 1
            time.sleep(0.5)  # Delay to avoid rate limiting
            if scraped_count >= 3000:  # Stop after 1000 posts (adjustable)
                logger.info("Reached target of 1000 posts.")
                break
        
        return scraped_count
    except requests.RequestException as e:
        logger.error(f"API request failed: {e}")
        return 0

def scrape_article_data_selenium(url, driver):
    """Scrape title, URL, date, and content from an article page using Selenium."""
    try:
        driver.get(url)
        time.sleep(2)  # Wait for page to load
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Extract title (from your snippet)
        title_tag = soup.find('h1')
        title = title_tag.get_text(strip=True) if title_tag else 'Untitled'
        
        # Extract date
        date_tag = soup.find('time', class_='entry-date') or soup.find('time') or soup.find('meta', property='article:published_time')
        date = date_tag.get_text(strip=True) or date_tag.get('content', 'No Date') if date_tag else 'No Date'
        
        # Extract text from all <p> tags (from your snippet)
        paragraphs = soup.find_all('p')
        full_text = ' '.join(p.get_text(strip=True) for p in paragraphs)
        words = full_text.split()
        content = ' '.join(words[:MAX_WORDS])
        
        logger.info(f"Scraped article: {title} from {url}")
        return title, url, date, content
    except Exception as e:
        logger.error(f"Failed to scrape {url}: {e}")
        return None, None, None, None

def get_article_links_selenium(page_url, base_url, driver):
    """Fetch article links using Selenium for JavaScript-rendered content."""
    try:
        driver.get(page_url)
        time.sleep(3)  # Wait for JavaScript to load
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Find article links
        articles = soup.find_all('article') or soup.find_all('div', class_=['post', 'article', 'entry'])
        links = []
        for article in articles:
            link_tag = article.find('a', class_='entry-title-link') or article.find('a', href=True)
            if link_tag and 'href' in link_tag.attrs:
                full_url = urljoin(base_url, link_tag['href'])
                links.append(full_url)
        
        # Find next page link
        next_page = soup.find('a', class_='next') or soup.find('a', text='Next') or soup.find('a', rel='next')
        next_url = urljoin(base_url, next_page['href']) if next_page and 'href' in next_page.attrs else None
        
        return links, next_url
    except Exception as e:
        logger.error(f"Error fetching {page_url} with Selenium: {e}")
        return [], None

def scrape_sitemap(base_url, driver):
    """Scrape article URLs from the website's sitemap."""
    sitemap_url = f"{base_url}/sitemap.xml"
    try:
        response = requests.get(sitemap_url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'xml')
        urls = [loc.text for loc in soup.find_all('loc') if '/post/' in loc.text or '/20' in loc.text]  # Filter for blog posts
        logger.info(f"Found {len(urls)} URLs in sitemap")
        return urls[:3000]  # Limit to 1000+ posts
    except requests.RequestException as e:
        logger.error(f"Failed to fetch sitemap {sitemap_url}: {e}")
        return []

def main():
    base_url = 'https://datascienceplus.com'
    output_file = 'datascienceplus_articles.csv'
    target_posts = 3000  # Target at least 1000 posts
    
    # Try API first
    logger.info("Attempting to scrape via WordPress REST API...")
    api_count = scrape_api(base_url)
    
    if api_count >= target_posts:
        logger.info(f"Scraped {api_count} articles via API, meeting target.")
    else:
        logger.info(f"API scraped {api_count} articles. Falling back to Selenium...")
        
        # Initialize Selenium
        options = Options()
        options.headless = True
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        
        try:
            # Try sitemap first
            all_links = set(scrape_sitemap(base_url, driver))
            logger.info(f"Collected {len(all_links)} unique URLs from sitemap")
            
            # If sitemap didn’t provide enough links, try pagination
            if len(all_links) < target_posts:
                page_url = base_url
                page_num = 1
                max_pages = 50  # Adjust to reach 1000+ posts
                while page_url and page_num <= max_pages and len(all_links) < target_posts:
                    logger.info(f"Scraping page {page_num}: {page_url}")
                    links, next_url = get_article_links_selenium(page_url, base_url, driver)
                    logger.info(f"Found {len(links)} article links on page {page_num}")
                    all_links.update(links)
                    page_url = next_url
                    page_num += 1
                    time.sleep(1)  # Delay between pages
            
            # Scrape articles
            scraped_count = api_count
            for article_url in all_links:
                # Check data size
                df = pd.DataFrame(data)
                if get_data_size(df) >= MAX_SIZE_BYTES:
                    logger.warning("Data size limit of 500 MB reached. Stopping scraping.")
                    break
                
                title, url, date, content = scrape_article_data_selenium(article_url, driver)
                if title and url and date and content:
                    data['Title'].append(title)
                    data['URL'].append(url)
                    data['Date'].append(date)
                    data['Content'].append(content)
                    scraped_count += 1
                    if scraped_count >= target_posts:
                        logger.info("Reached target of 1000 posts.")
                        break
                time.sleep(0.5)  # Delay between articles
            
            logger.info(f"Total unique article links found: {len(all_links)}")
        
        finally:
            driver.quit()
    
    # Create DataFrame
    df = pd.DataFrame(data)
    logger.info(f"Total articles scraped: {len(df)}")
    
    # Save to CSV
    if not df.empty:
        df.to_csv(output_file, index=False)
        file_size = os.path.getsize(output_file) / (1024 * 1024)  # Size in MB
        logger.info(f"Data saved to {output_file}. CSV file size: {file_size:.2f} MB")
    else:
        logger.warning("No data was scraped. CSV file not created.")

if __name__ == '__main__':
    main()

2025-07-24 11:28:02,220 - INFO - Attempting to scrape via WordPress REST API...
2025-07-24 11:28:06,484 - INFO - Scraped via API: Linking R and Python to retrieve financial data and plot a candlestick
2025-07-24 11:28:06,741 - INFO - Scraped via API: An R alternative to pairs for -omics QC
2025-07-24 11:28:06,761 - INFO - Scraped via API: Visualizing economic data with pretty worldmaps
2025-07-24 11:28:06,793 - INFO - Scraped via API: How to Incorporate ML.Net With Algorithmic Trading
2025-07-24 11:28:06,820 - INFO - Scraped via API: How I selected my starting word for Wordle using simulations and R
2025-07-24 11:28:06,834 - INFO - Scraped via API: Forecast using Arima Model in R
2025-07-24 11:28:06,864 - INFO - Scraped via API: Propagating nerve impulse in  Hodgkin-Huxley model. Modeling with R. Part 2
2025-07-24 11:28:06,887 - INFO - Scraped via API: Topic Modeling and Latent Dirichlet Allocation (LDA)
2025-07-24 11:28:06,921 - INFO - Scraped via API: Ditch p-values. Use Bootstrap co

In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from tqdm import tqdm
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Define headers to mimic a browser
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Keywords to filter ML/AI articles
ml_ai_keywords = [
    'machine learning', 'artificial intelligence', 'deep learning', 'neural network',
    'data science', 'nlp', 'natural language processing', 'computer vision',
    'reinforcement learning', 'time series', 'recommendation system', 'arima', 'prophet',
    'tensorflow', 'pytorch', 'scikit-learn', 'gradient boosting', 'xgboost'
]

# Function to check if an article is ML/AI-related
def is_ml_ai_article(text):
    text = text.lower()
    return any(keyword in text for keyword in ml_ai_keywords)

# Function to scrape a single blog page
def scrape_page(url):
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()  # Raise exception for bad status codes
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Find article containers (adjust selector based on site structure)
        articles = soup.find_all('div', class_='list-card-content')  # Common class for blog posts
        page_articles = []
        
        for article in articles:
            try:
                # Extract title
                title_tag = article.find('h4') or article.find('h2')
                title = title_tag.text.strip() if title_tag else 'No Title'
                
                # Skip if not ML/AI-related
                if not is_ml_ai_article(title):
                    continue
                
                # Extract URL
                url_tag = article.find('a', href=True)
                article_url = url_tag['href'] if url_tag else None
                if not article_url.startswith('http'):
                    article_url = f"https://www.analyticsvidhya.com{article_url}"
                
                # Extract summary (short description)
                summary_tag = article.find('div', class_='list-card-excerpt') or article.find('p')
                summary = summary_tag.text.strip() if summary_tag else 'No Summary'
                
                # Extract publication date
                date_tag = article.find('time') or article.find('span', class_='post-date')
                pub_date = date_tag.text.strip() if date_tag else 'No Date'
                
                # Fetch full article content
                content = 'No Content'
                if article_url:
                    try:
                        article_response = requests.get(article_url, headers=headers, timeout=10)
                        article_response.raise_for_status()
                        article_soup = BeautifulSoup(article_response.content, 'html.parser')
                        
                        # Extract main content (adjust selector based on site structure)
                        content_div = article_soup.find('div', class_='post-content') or \
                                     article_soup.find('article')
                        if content_div:
                            # Remove scripts, styles, and unwanted elements
                            for unwanted in content_div(['script', 'style', 'aside']):
                                unwanted.decompose()
                            content = content_div.get_text(strip=True, separator=' ')
                            # Limit content length to avoid excessive data
                            content = content[:5000]
                        
                        # Verify ML/AI relevance with content
                        if not is_ml_ai_article(content):
                            continue
                    except requests.RequestException as e:
                        logging.warning(f"Failed to fetch article content for {article_url}: {e}")
                
                page_articles.append({
                    'title': title,
                    'url': article_url,
                    'summary': summary,
                    'pub_date': pub_date,
                    'content': content
                })
                
            except Exception as e:
                logging.warning(f"Error processing article: {e}")
                continue
        
        return page_articles
    
    except requests.RequestException as e:
        logging.error(f"Error fetching page {url}: {e}")
        return []

# Main scraping function
def scrape_analytics_vidhya_articles(target_count=1000):
    articles = []
    page = 1
    base_url = "https://www.analyticsvidhya.com/blog/page/{}/"
    
    # Progress bar
    with tqdm(total=target_count, desc="Scraping Articles") as pbar:
        while len(articles) < target_count:
            url = base_url.format(page)
            logging.info(f"Scraping page {page}: {url}")
            
            page_articles = scrape_page(url)
            if not page_articles:
                logging.info(f"No more articles found on page {page}. Stopping.")
                break
            
            articles.extend(page_articles)
            pbar.update(len(page_articles))
            
            # Check if we've reached the target
            if len(articles) >= target_count:
                articles = articles[:target_count]  # Trim to exact count
                break
            
            page += 1
            time.sleep(2)  # Respectful delay to avoid overwhelming server
            
            # Check for pagination limit (e.g., no more pages)
            if page > 100:  # Arbitrary limit; adjust based on site
                logging.info("Reached page limit. Stopping.")
                break
    
    return articles

# Save articles to CSV
def save_to_csv(articles, filename='ml_ai_articles.csv'):
    df = pd.DataFrame(articles)
    df.to_csv(filename, index=False, encoding='utf-8')
    logging.info(f"Saved {len(articles)} articles to {filename}")

# Execute scraping
if __name__ == "__main__":
    try:
        # Check robots.txt (manual step recommended)
        robots_url = "https://www.analyticsvidhya.com/robots.txt"
        robots_response = requests.get(robots_url, headers=headers)
        if 'Disallow: /blog/' in robots_response.text:
            logging.error("Scraping blog pages is disallowed by robots.txt. Aborting.")
        else:
            articles = scrape_analytics_vidhya_articles(target_count=1000)
            if articles:
                save_to_csv(articles)
                logging.info(f"Successfully scraped {len(articles)} ML/AI articles.")
            else:
                logging.info("No articles scraped.")
    except Exception as e:
        logging.error(f"Script failed: {e}")
        

2025-07-24 11:57:50,936 - ERROR - Scraping blog pages is disallowed by robots.txt. Aborting.


In [14]:
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import re
from tqdm import tqdm
import logging
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Define headers for requests
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Expanded keywords for ML/AI filtering
ml_ai_keywords = [
    'machine learning', 'artificial intelligence', 'deep learning', 'neural network',
    'data science', 'nlp', 'natural language processing', 'computer vision',
    'reinforcement learning', 'time series', 'recommendation system', 'arima', 'prophet',
    'tensorflow', 'pytorch', 'scikit-learn', 'gradient boosting', 'xgboost',
    'llm', 'large language model', 'ai', 'predictive modeling', 'clustering', 'classification',
    'genai', 'model training', 'feature engineering', 'supervised learning', 'unsupervised learning'
]

# Function to check if an article is ML/AI-related
def is_ml_ai_article(text):
    text = text.lower() if text else ''
    return any(keyword in text for keyword in ml_ai_keywords)

# Set up Selenium driver
def setup_selenium():
    chrome_options = Options()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument(f'user-agent={headers["User-Agent"]}')
    driver = webdriver.Chrome(options=chrome_options)
    driver.set_page_load_timeout(30)  # Increase timeout
    return driver

# Set up requests with retries
session = requests.Session()
retries = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
session.mount('https://', HTTPAdapter(max_retries=retries))

# Site configurations
site_configs = [
    {
        'name': 'Towards Data Science',
        'base_url': 'https://towardsdatascience.com/page/{}',
        'article_selector': 'div.u-lineHeightBase.postItem',
        'title_selector': 'h3,h2',
        'summary_selector': 'div.u-lineHeightTight,p',
        'date_selector': 'time,span[class*="date"]',
        'content_selector': 'div.postArticle-content,article',
        'use_selenium': True
    },
    {
        'name': 'KDnuggets',
        'base_url': 'https://www.kdnuggets.com/topics/machine-learning/page/{}',
        'article_selector': 'li.post,li[class*="post"],article,div[class*="post"]',
        'title_selector': 'h3,h2,h4,a[class*="title"]',
        'summary_selector': 'div.excerpt,div[class*="excerpt"],p',
        'date_selector': 'time,span[class*="date"],div[class*="meta"]',
        'content_selector': 'div.content,div[class*="content"],article,div[class*="post"]',
        'use_selenium': True,
        'fallback_urls': ['https://www.kdnuggets.com/news/page/{}', 'https://www.kdnuggets.com/topics/artificial-intelligence/page/{}']
    },
    {
        'name': 'Analytics Vidhya',
        'base_url': 'https://www.analyticsvidhya.com/blog/page/{}',
        'article_selector': 'div.list-card-content,article',
        'title_selector': 'h4,h2,h3',
        'summary_selector': 'div.list-card-excerpt,p',
        'date_selector': 'time,span[class*="post-date"]',
        'content_selector': 'div.post-content,article',
        'use_selenium': False
    },
    {
        'name': 'Machine Learning Mastery',
        'base_url': 'https://machinelearningmastery.com/page/{}',
        'article_selector': 'article.post',
        'title_selector': 'h2.entry-title,h1,h3',
        'summary_selector': 'div.entry-summary,p',
        'date_selector': 'time.entry-date,span[class*="date"]',
        'content_selector': 'div.entry-content,article',
        'use_selenium': False
    },
    {
        'name': 'OpenML',
        'base_url': 'https://www.openml.org/news/page/{}',
        'article_selector': 'div.card,article',
        'title_selector': 'h5.card-title,h3,h2',
        'summary_selector': 'p.card-text,p',
        'date_selector': 'span.date,time',
        'content_selector': 'div.card-body,article',
        'use_selenium': False
    },
    {
        'name': 'Papers With Code',
        'base_url': 'https://paperswithcode.com/news?page={}',
        'article_selector': 'div.paper-card,article',
        'title_selector': 'h5,h3,h2',
        'summary_selector': 'p.paper-abstract,p',
        'date_selector': 'span.date,time',
        'content_selector': 'div.paper-content,article',
        'use_selenium': True
    },
    {
        'name': 'Data Science Central',
        'base_url': 'https://www.datasciencecentral.com/page/{}',
        'article_selector': 'div.post,article',
        'title_selector': 'h2.post-title,h3,h1',
        'summary_selector': 'div.post-excerpt,p',
        'date_selector': 'span.post-date,time',
        'content_selector': 'div.post-content,article',
        'use_selenium': False
    },
    {
        'name': 'WIRED',
        'base_url': 'https://www.wired.com/tag/artificial-intelligence/page/{}',
        'article_selector': 'div.card,article',
        'title_selector': 'h3,h2',
        'summary_selector': 'p,div[class*="summary"]',
        'date_selector': 'time,span[class*="date"]',
        'content_selector': 'div.article__body,article',
        'use_selenium': True
    },
    {
        'name': 'Nature',
        'base_url': 'https://www.nature.com/subjects/machine-learning/articles?page={}',
        'article_selector': 'article',
        'title_selector': 'h3,h2',
        'summary_selector': 'p.article__teaser,p',
        'date_selector': 'time',
        'content_selector': 'div.article__body,article',
        'use_selenium': False
    }
]

# Function to scrape a single page
def scrape_page(url, driver, config, use_selenium=True):
    try:
        if use_selenium:
            driver.get(url)
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, config['article_selector'])))
            soup = BeautifulSoup(driver.page_source, 'html.parser')
        else:
            response = session.get(url, headers=headers, timeout=15)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')
        
        articles = soup.select(config['article_selector'])
        logging.info(f"Found {len(articles)} article containers on {url} ({config['name']})")
        
        page_articles = []
        for article in articles:
            try:
                title_tag = article.select_one(config['title_selector'])
                title = title_tag.text.strip() if title_tag else 'No Title'
                
                url_tag = article.find('a', href=True)
                article_url = url_tag['href'] if url_tag else None
                if article_url and not article_url.startswith('http'):
                    article_url = f"https://{config['base_url'].split('/')[2]}{article_url}"
                
                summary_tag = article.select_one(config['summary_selector'])
                summary = summary_tag.text.strip() if summary_tag else 'No Summary'
                
                date_tag = article.select_one(config['date_selector'])
                pub_date = date_tag.text.strip() if date_tag else 'No Date'
                
                if not is_ml_ai_article(title):
                    logging.debug(f"Skipping article (title not ML/AI): {title} ({config['name']})")
                    continue
                
                content = 'No Content'
                if article_url:
                    for attempt in range(2):  # Retry once
                        try:
                            if use_selenium:
                                driver.get(article_url)
                                WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, config['content_selector'])))
                                article_soup = BeautifulSoup(driver.page_source, 'html.parser')
                            else:
                                article_response = session.get(article_url, headers=headers, timeout=15)
                                article_response.raise_for_status()
                                article_soup = BeautifulSoup(article_response.content, 'html.parser')
                            
                            content_div = article_soup.select_one(config['content_selector'])
                            if content_div:
                                for unwanted in content_div(['script', 'style', 'aside', 'footer', 'nav']):
                                    unwanted.decompose()
                                content = content_div.get_text(strip=True, separator=' ')[:5000]
                            
                            if content == 'No Content' and is_ml_ai_article(title):
                                logging.debug(f"Content fetch failed for {title}, but title is ML/AI-related ({config['name']})")
                            elif not is_ml_ai_article(content):
                                logging.debug(f"Skipping article (content not ML/AI): {title} ({config['name']})")
                                continue
                            break
                        except Exception as e:
                            logging.warning(f"Attempt {attempt+1} failed to fetch content for {article_url}: {e}")
                            content = 'Content fetch failed'
                            if is_ml_ai_article(title):
                                break
                            time.sleep(1)
                
                page_articles.append({
                    'title': title,
                    'url': article_url,
                    'summary': summary,
                    'pub_date': pub_date,
                    'content': content,
                    'source': config['name']
                })
                logging.debug(f"Added article from {config['name']}: {title}")
                
            except Exception as e:
                logging.warning(f"Error processing article from {config['name']}: {e}")
                continue
        
        return page_articles
    
    except Exception as e:
        logging.error(f"Error fetching page {url} ({config['name']}): {e}")
        return []

# Main scraping function
def scrape_all_sites(target_count=1000):
    articles = []
    driver = setup_selenium()
    try:
        for config in site_configs:
            if len(articles) >= target_count:
                break
            
            urls_to_try = [config['base_url']] + config.get('fallback_urls', [])
            for base_url in urls_to_try:
                if len(articles) >= target_count:
                    break
                
                page = 1
                logging.info(f"Scraping {config['name']} starting at {base_url.format(page)}")
                
                robots_url = f"https://{base_url.split('/')[2]}/robots.txt"
                try:
                    robots_response = session.get(robots_url, headers=headers, timeout=10)
                    if 'Disallow: ' in robots_response.text and any(
                        base_url.split('/')[3] in line for line in robots_response.text.splitlines()
                    ):
                        logging.warning(f"Scraping disallowed by robots.txt for {config['name']} ({base_url}). Skipping.")
                        continue
                except Exception:
                    logging.warning(f"Could not check robots.txt for {config['name']}. Proceeding cautiously.")
                
                while len(articles) < target_count:
                    url = base_url.format(page)
                    page_articles = scrape_page(url, driver, config, use_selenium=config['use_selenium'])
                    logging.info(f"Collected {len(page_articles)} articles from page {page} ({config['name']}). Total: {len(articles) + len(page_articles)}")
                    
                    articles.extend(page_articles)
                    
                    if len(articles) >= target_count:
                        articles = articles[:target_count]
                        break
                    
                    if not page_articles and page > 1:
                        logging.info(f"No more articles on page {page} for {config['name']} ({base_url}). Trying next URL.")
                        break
                    
                    page += 1
                    time.sleep(2)
                    
                    if page > 100:
                        logging.info(f"Reached page limit for {config['name']} ({base_url}). Trying next URL.")
                        break
        
        return articles
    finally:
        driver.quit()

# Save articles to CSV
def save_to_csv(articles, filename='ml_ai_articles.csv'):
    df = pd.DataFrame(articles)
    df.to_csv(filename, index=False, encoding='utf-8')
    logging.info(f"Saved {len(articles)} articles to {filename}")

# Execute scraping
if __name__ == "__main__":
    try:
        articles = scrape_all_sites(target_count=1000)
        if articles:
            save_to_csv(articles)
            logging.info(f"Successfully scraped {len(articles)} ML/AI articles across all sites.")
        else:
            logging.info("No articles scraped.")
    except Exception as e:
        logging.error(f"Script failed: {e}")

2025-07-24 12:44:40,237 - INFO - Scraping Towards Data Science starting at https://towardsdatascience.com/page/1
2025-07-24 12:44:51,752 - ERROR - Error fetching page https://towardsdatascience.com/page/1 (Towards Data Science): Message: 
Stacktrace:
	GetHandleVerifier [0x0x7ff636a0e935+77845]
	GetHandleVerifier [0x0x7ff636a0e990+77936]
	(No symbol) [0x0x7ff6367c9cda]
	(No symbol) [0x0x7ff6368206aa]
	(No symbol) [0x0x7ff63682095c]
	(No symbol) [0x0x7ff636873d07]
	(No symbol) [0x0x7ff63684890f]
	(No symbol) [0x0x7ff636870b07]
	(No symbol) [0x0x7ff6368486a3]
	(No symbol) [0x0x7ff636811791]
	(No symbol) [0x0x7ff636812523]
	GetHandleVerifier [0x0x7ff636ce684d+3059501]
	GetHandleVerifier [0x0x7ff636ce0c0d+3035885]
	GetHandleVerifier [0x0x7ff636d00400+3164896]
	GetHandleVerifier [0x0x7ff636a28c3e+185118]
	GetHandleVerifier [0x0x7ff636a3054f+216111]
	GetHandleVerifier [0x0x7ff636a172e4+113092]
	GetHandleVerifier [0x0x7ff636a17499+113529]
	GetHandleVerifier [0x0x7ff6369fe298+10616]
	BaseThread

In [15]:
pip install feedparser 

  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6070 sha256=b95147c1587060ab31793e6043612ec9b792cd84c8bd4e344eafb2953fe45721
  Stored in directory: c:\users\user\appdata\local\pip\cache\wheels\83\63\2f\117884c3b19d46b64d3d61690333aa80c88dc14050e269c546Note: you may need to restart the kernel to use updated packages.
Successfully built sgmllib3k



In [16]:
import feedparser
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import re
from tqdm import tqdm
import logging
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Define headers for requests
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Expanded keywords for ML/AI filtering
ml_ai_keywords = [
    'machine learning', 'artificial intelligence', 'deep learning', 'neural network',
    'data science', 'nlp', 'natural language processing', 'computer vision',
    'reinforcement learning', 'time series', 'recommendation system', 'arima', 'prophet',
    'tensorflow', 'pytorch', 'scikit-learn', 'gradient boosting', 'xgboost',
    'llm', 'large language model', 'ai', 'predictive modeling', 'clustering', 'classification',
    'genai', 'model training', 'feature engineering', 'supervised learning', 'unsupervised learning'
]

# Function to check if an article is ML/AI-related
def is_ml_ai_article(text):
    text = text.lower() if text else ''
    return any(keyword in text for keyword in ml_ai_keywords)

# Set up Selenium driver
def setup_selenium():
    chrome_options = Options()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument(f'user-agent={headers["User-Agent"]}')
    driver = webdriver.Chrome(options=chrome_options)
    driver.set_page_load_timeout(30)
    return driver

# Set up requests with retries
session = requests.Session()
retries = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
session.mount('https://', HTTPAdapter(max_retries=retries))

# Site configurations with RSS feeds and web scraping fallbacks
site_configs = [
    {
        'name': 'Towards Data Science',
        'feed_url': 'https://towardsdatascience.com/feed',
        'base_url': 'https://towardsdatascience.com/page/{}',
        'article_selector': 'div.u-lineHeightBase.postItem',
        'title_selector': 'h3,h2',
        'summary_selector': 'div.u-lineHeightTight,p',
        'date_selector': 'time,span[class*="date"]',
        'content_selector': 'div.postArticle-content,article',
        'use_selenium': True
    },
    {
        'name': 'KDnuggets',
        'feed_url': 'https://www.kdnuggets.com/feed',
        'base_url': 'https://www.kdnuggets.com/topics/machine-learning/page/{}',
        'article_selector': 'li.post,li[class*="post"],article,div[class*="post"]',
        'title_selector': 'h3,h2,h4,a[class*="title"]',
        'summary_selector': 'div.excerpt,div[class*="excerpt"],p',
        'date_selector': 'time,span[class*="date"],div[class*="meta"]',
        'content_selector': 'div.content,div[class*="content"],article,div[class*="post"]',
        'use_selenium': True,
        'fallback_urls': ['https://www.kdnuggets.com/news/page/{}', 'https://www.kdnuggets.com/topics/artificial-intelligence/page/{}']
    },
    {
        'name': 'Analytics Vidhya',
        'feed_url': 'https://www.analyticsvidhya.com/blog/feed/',
        'base_url': 'https://www.analyticsvidhya.com/blog/page/{}',
        'article_selector': 'div.list-card-content,article',
        'title_selector': 'h4,h2,h3',
        'summary_selector': 'div.list-card-excerpt,p',
        'date_selector': 'time,span[class*="post-date"]',
        'content_selector': 'div.post-content,article',
        'use_selenium': False
    },
    {
        'name': 'Machine Learning Mastery',
        'feed_url': 'https://machinelearningmastery.com/feed/',
        'base_url': 'https://machinelearningmastery.com/page/{}',
        'article_selector': 'article.post',
        'title_selector': 'h2.entry-title,h1,h3',
        'summary_selector': 'div.entry-summary,p',
        'date_selector': 'time.entry-date,span[class*="date"]',
        'content_selector': 'div.entry-content,article',
        'use_selenium': False
    },
    {
        'name': 'OpenML',
        'feed_url': 'https://www.openml.org/news/feed',
        'base_url': 'https://www.openml.org/news/page/{}',
        'article_selector': 'div.card,article',
        'title_selector': 'h5.card-title,h3,h2',
        'summary_selector': 'p.card-text,p',
        'date_selector': 'span.date,time',
        'content_selector': 'div.card-body,article',
        'use_selenium': False
    },
    {
        'name': 'Papers With Code',
        'feed_url': None,  # No RSS feed; use web scraping
        'base_url': 'https://paperswithcode.com/news?page={}',
        'article_selector': 'div.paper-card,article',
        'title_selector': 'h5,h3,h2',
        'summary_selector': 'p.paper-abstract,p',
        'date_selector': 'span.date,time',
        'content_selector': 'div.paper-content,article',
        'use_selenium': True
    },
    {
        'name': 'Data Science Central',
        'feed_url': 'https://www.datasciencecentral.com/feed/',
        'base_url': 'https://www.datasciencecentral.com/page/{}',
        'article_selector': 'div.post,article',
        'title_selector': 'h2.post-title,h3,h1',
        'summary_selector': 'div.post-excerpt,p',
        'date_selector': 'span.post-date,time',
        'content_selector': 'div.post-content,article',
        'use_selenium': False
    },
    {
        'name': 'WIRED',
        'feed_url': 'https://www.wired.com/feed/tag/artificial-intelligence/rss',
        'base_url': 'https://www.wired.com/tag/artificial-intelligence/page/{}',
        'article_selector': 'div.card,article',
        'title_selector': 'h3,h2',
        'summary_selector': 'p,div[class*="summary"]',
        'date_selector': 'time,span[class*="date"]',
        'content_selector': 'div.article__body,article',
        'use_selenium': True
    },
    {
        'name': 'Nature',
        'feed_url': 'https://www.nature.com/subjects/machine-learning.rss',
        'base_url': 'https://www.nature.com/subjects/machine-learning/articles?page={}',
        'article_selector': 'article',
        'title_selector': 'h3,h2',
        'summary_selector': 'p.article__teaser,p',
        'date_selector': 'time',
        'content_selector': 'div.article__body,article',
        'use_selenium': False
    }
]

# Function to scrape a single feed
def scrape_feed(feed_url, site_name):
    try:
        feed = feedparser.parse(feed_url)
        if not feed.entries:
            logging.warning(f"No entries found in feed for {site_name} ({feed_url})")
            return []
        
        articles = []
        for entry in feed.entries:
            try:
                title = entry.get('title', 'No Title')
                if not is_ml_ai_article(title):
                    logging.debug(f"Skipping feed entry (title not ML/AI): {title} ({site_name})")
                    continue
                
                article_url = entry.get('link', None)
                summary = entry.get('summary', entry.get('description', 'No Summary'))
                pub_date = entry.get('published', entry.get('updated', 'No Date'))
                
                content = 'No Content'
                if article_url:
                    try:
                        response = session.get(article_url, headers=headers, timeout=15)
                        response.raise_for_status()
                        soup = BeautifulSoup(response.content, 'html.parser')
                        content_div = soup.select_one('article,div[class*="content"],div[class*="post"]')
                        if content_div:
                            for unwanted in content_div(['script', 'style', 'aside', 'footer', 'nav']):
                                unwanted.decompose()
                            content = content_div.get_text(strip=True, separator=' ')[:5000]
                        
                        if content == 'No Content' and is_ml_ai_article(title):
                            logging.debug(f"Content fetch failed for {title}, but title is ML/AI-related ({site_name})")
                        elif not is_ml_ai_article(content):
                            logging.debug(f"Skipping feed entry (content not ML/AI): {title} ({site_name})")
                            continue
                    except Exception as e:
                        logging.warning(f"Failed to fetch content for {article_url}: {e}")
                        if is_ml_ai_article(title):
                            content = 'Content fetch failed'
                        else:
                            continue
                
                articles.append({
                    'title': title,
                    'url': article_url,
                    'summary': summary,
                    'pub_date': pub_date,
                    'content': content,
                    'source': site_name
                })
                logging.debug(f"Added feed entry from {site_name}: {title}")
                
            except Exception as e:
                logging.warning(f"Error processing feed entry from {site_name}: {e}")
                continue
        
        return articles
    
    except Exception as e:
        logging.error(f"Error parsing feed {feed_url} ({site_name}): {e}")
        return []

# Function to scrape a single page (web scraping fallback)
def scrape_page(url, driver, config, use_selenium=True):
    try:
        if use_selenium:
            driver.get(url)
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, config['article_selector'])))
            soup = BeautifulSoup(driver.page_source, 'html.parser')
        else:
            response = session.get(url, headers=headers, timeout=15)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')
        
        articles = soup.select(config['article_selector'])
        logging.info(f"Found {len(articles)} article containers on {url} ({config['name']})")
        
        page_articles = []
        for article in articles:
            try:
                title_tag = article.select_one(config['title_selector'])
                title = title_tag.text.strip() if title_tag else 'No Title'
                
                if not is_ml_ai_article(title):
                    logging.debug(f"Skipping article (title not ML/AI): {title} ({config['name']})")
                    continue
                
                url_tag = article.find('a', href=True)
                article_url = url_tag['href'] if url_tag else None
                if article_url and not article_url.startswith('http'):
                    article_url = f"https://{config['base_url'].split('/')[2]}{article_url}"
                
                summary_tag = article.select_one(config['summary_selector'])
                summary = summary_tag.text.strip() if summary_tag else 'No Summary'
                
                date_tag = article.select_one(config['date_selector'])
                pub_date = date_tag.text.strip() if date_tag else 'No Date'
                
                content = 'No Content'
                if article_url:
                    for attempt in range(2):
                        try:
                            if use_selenium:
                                driver.get(article_url)
                                WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, config['content_selector'])))
                                article_soup = BeautifulSoup(driver.page_source, 'html.parser')
                            else:
                                article_response = session.get(article_url, headers=headers, timeout=15)
                                article_response.raise_for_status()
                                article_soup = BeautifulSoup(article_response.content, 'html.parser')
                            
                            content_div = article_soup.select_one(config['content_selector'])
                            if content_div:
                                for unwanted in content_div(['script', 'style', 'aside', 'footer', 'nav']):
                                    unwanted.decompose()
                                content = content_div.get_text(strip=True, separator=' ')[:5000]
                            
                            if content == 'No Content' and is_ml_ai_article(title):
                                logging.debug(f"Content fetch failed for {title}, but title is ML/AI-related ({config['name']})")
                            elif not is_ml_ai_article(content):
                                logging.debug(f"Skipping article (content not ML/AI): {title} ({config['name']})")
                                continue
                            break
                        except Exception as e:
                            logging.warning(f"Attempt {attempt+1} failed to fetch content for {article_url}: {e}")
                            content = 'Content fetch failed'
                            if is_ml_ai_article(title):
                                break
                            time.sleep(1)
                
                page_articles.append({
                    'title': title,
                    'url': article_url,
                    'summary': summary,
                    'pub_date': pub_date,
                    'content': content,
                    'source': config['name']
                })
                logging.debug(f"Added article from {config['name']}: {title}")
                
            except Exception as e:
                logging.warning(f"Error processing article from {config['name']}: {e}")
                continue
        
        return page_articles
    
    except Exception as e:
        logging.error(f"Error fetching page {url} ({config['name']}): {e}")
        return []

# Main scraping function
def scrape_all_sites(target_count=1000):
    articles = []
    driver = setup_selenium()
    try:
        for config in site_configs:
            if len(articles) >= target_count:
                break
            
            # Try RSS feed first
            if config.get('feed_url'):
                logging.info(f"Scraping feed for {config['name']}: {config['feed_url']}")
                feed_articles = scrape_feed(config['feed_url'], config['name'])
                articles.extend(feed_articles[:target_count - len(articles)])
                logging.info(f"Collected {len(feed_articles)} articles from feed for {config['name']}. Total: {len(articles)}")
                
                if len(articles) >= target_count:
                    break
            
            # Fallback to web scraping
            urls_to_try = [config['base_url']] + config.get('fallback_urls', [])
            for base_url in urls_to_try:
                if len(articles) >= target_count:
                    break
                
                page = 1
                logging.info(f"Scraping web for {config['name']} starting at {base_url.format(page)}")
                
                robots_url = f"https://{base_url.split('/')[2]}/robots.txt"
                try:
                    robots_response = session.get(robots_url, headers=headers, timeout=10)
                    if 'Disallow: ' in robots_response.text and any(
                        base_url.split('/')[3] in line for line in robots_response.text.splitlines()
                    ):
                        logging.warning(f"Scraping disallowed by robots.txt for {config['name']} ({base_url}). Skipping.")
                        continue
                except Exception:
                    logging.warning(f"Could not check robots.txt for {config['name']}. Proceeding cautiously.")
                
                while len(articles) < target_count:
                    url = base_url.format(page)
                    page_articles = scrape_page(url, driver, config, use_selenium=config['use_selenium'])
                    logging.info(f"Collected {len(page_articles)} articles from page {page} ({config['name']}). Total: {len(articles) + len(page_articles)}")
                    
                    articles.extend(page_articles)
                    
                    if len(articles) >= target_count:
                        articles = articles[:target_count]
                        break
                    
                    if not page_articles and page > 1:
                        logging.info(f"No more articles on page {page} for {config['name']} ({base_url}). Trying next URL.")
                        break
                    
                    page += 1
                    time.sleep(2)
                    
                    if page > 100:
                        logging.info(f"Reached page limit for {config['name']} ({base_url}). Trying next URL.")
                        break
        
        return articles
    finally:
        driver.quit()

# Save articles to CSV
def save_to_csv(articles, filename='ml_ai_articles.csv'):
    df = pd.DataFrame(articles)
    df.to_csv(filename, index=False, encoding='utf-8')
    logging.info(f"Saved {len(articles)} articles to {filename}")

# Execute scraping
if __name__ == "__main__":
    try:
        articles = scrape_all_sites(target_count=1000)
        if articles:
            save_to_csv(articles)
            logging.info(f"Successfully scraped {len(articles)} ML/AI articles across all sites.")
        else:
            logging.info("No articles scraped.")
    except Exception as e:
        logging.error(f"Script failed: {e}")

2025-07-24 17:41:24,542 - INFO - Scraping feed for Towards Data Science: https://towardsdatascience.com/feed
2025-07-24 17:41:26,920 - WARNING - Failed to fetch content for https://towardsdatascience.com/why-bi-in-the-ai-age/: 403 Client Error: Forbidden for url: https://towardsdatascience.com/why-bi-in-the-ai-age/
2025-07-24 17:41:27,655 - WARNING - Failed to fetch content for https://towardsdatascience.com/torchvista-building-an-interactive-pytorch-visualization-package-for-notebooks/: 403 Client Error: Forbidden for url: https://towardsdatascience.com/torchvista-building-an-interactive-pytorch-visualization-package-for-notebooks/
2025-07-24 17:41:28,615 - WARNING - Failed to fetch content for https://towardsdatascience.com/when-llms-try-to-reason-experiments-in-text-and-vision-based-abstraction/: 403 Client Error: Forbidden for url: https://towardsdatascience.com/when-llms-try-to-reason-experiments-in-text-and-vision-based-abstraction/
2025-07-24 17:41:29,173 - WARNING - Failed to f